# Overlap-Hybrid Atlas DeePC Reacher Pipeline

This notebook implements a softer alternative to hard one-chart atlas DeePC.

The method is:

1. collect/load stabilizing offline Reacher rollouts;
2. partition the offline Hankel columns into prediction-error charts;
3. compress each chart with an SVD subspace basis;
4. online, select the top relevant charts and solve one DeePC problem over their combined bounded subspace;
5. execute the DeePC nominal action blended with a small IK-consistent stabilizing correction.

The key difference from the previous hard-atlas controller is that the online optimizer is allowed to use an overlap of local chart subspaces, not exactly one chart.


In [ ]:
from pathlib import Path
import json
import os

os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib").resolve()))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np

import expert_pipeline_search as pipe

PROJECT_DIR = Path.cwd()
HANKEL_PATH = PROJECT_DIR / "stabilizing_offline_hankel.npy"
LABEL_PATH = PROJECT_DIR / "stabilizing_prediction_error_labels.npz"
RESULT_DIR = PROJECT_DIR / "overlap_hybrid_results"
SUMMARY_PATH = RESULT_DIR / "summary.json"
RESULT_DIR.mkdir(exist_ok=True)

SUBSPACE_ENERGY = 0.999
SUBSPACE_RANK = 90
EVAL_SEEDS = [1, 2, 3, 4, 5, 42, 7, 8]


## 1. Offline Data

This pipeline intentionally uses the stabilizing offline dataset. It is not the fair IID Select-DeePC data experiment. The goal here is to test whether atlas/subspace DeePC can reuse a library of successful behavior across different targets.


In [ ]:
if HANKEL_PATH.exists():
    H = np.load(HANKEL_PATH)
    print("Loaded offline Hankel:", H.shape)
else:
    H = pipe.collect_expert_hankel(seed=123, episodes=140, episode_len=180, limit=0.5, noise_std=0.035)
    np.save(HANKEL_PATH, H)
    print("Collected offline Hankel:", H.shape)


## 2. Prediction-Error Atlas

The partition is still prediction-error based. The labels are checked against the current Hankel split dimensions so stale labels do not silently get reused.


In [ ]:
def labels_match_current_split(path):
    if not path.exists():
        return False
    data = np.load(path, allow_pickle=True)
    required = {"labels", "t_ini", "t_f", "l", "m", "p"}
    if not required.issubset(data.files):
        return False
    return (
        int(data["t_ini"]) == pipe.T_INI
        and int(data["t_f"]) == pipe.T_F
        and int(data["l"]) == pipe.L
        and int(data["m"]) == pipe.M
        and int(data["p"]) == pipe.P
    )

if labels_match_current_split(LABEL_PATH):
    labels = np.load(LABEL_PATH, allow_pickle=True)["labels"]
    print("Loaded labels:", np.bincount(labels.astype(int), minlength=pipe.K).tolist())
else:
    labels = pipe.learn_prediction_error_atlas(H, seed=0, label_path=LABEL_PATH)
    print("Learned labels:", np.bincount(labels.astype(int), minlength=pipe.K).tolist())


## 3. Overlap-Hybrid Controller

The online controller uses `mode="mixture_subspace"` and `chart_selection="mixture_relevance"`.

At each step it selects the two most relevant charts, concatenates their bounded SVD bases, and solves DeePC once over the combined subspace. The final action is a mostly-DeePC action plus a small stabilizing correction:

`u = 0.8 * u_deepc + 0.2 * u_stabilizing`

This correction is deliberately small. It is there to handle chart-overlap and local damping, while DeePC still supplies the nominal atlas/subspace behavior.


In [ ]:
BASE_VARIANT = dict(
    name="overlap_hybrid",
    mode="mixture_subspace",
    rank=SUBSPACE_RANK,
    energy=SUBSPACE_ENERGY,
    try_top=2,
    mixture_top=2,
    max_cols=500,
    u_limit=0.5,
    q_xy=500000.0,
    q_dq=1000.0,
    q_joint=1000.0,
    q_sum=0.0,
    q_terminal_xy=2000000.0,
    q_terminal_dq=1000.0,
    q_terminal_joint=0.0,
    q_terminal_sum=0.0,
    r=0.001,
    lam_g=0.05,
    lam_ini=100000.0,
    lam_du=3.0,
    solver="CLARABEL",
    steps=220,
    warmup_scale=0.0,
    chart_selection="mixture_relevance",
    relevance_past_weight=0.1,
    chart_switch_cost=0.0,
    coef_bound_scale=0.75,
    coef_bound_affine=False,
    fixed_chart=None,
    use_ik_ref=True,
    ik_elbow="down",
    feedback_blend=0.2,
    feedback_kp=6.0,
    feedback_kd=0.2,
    feedback_fallback=True,
)


## 4. Multi-Target Evaluation


In [ ]:
summaries = []
for seed in EVAL_SEEDS:
    variant = pipe.Variant(**{**BASE_VARIANT, "name": f"overlap_hybrid_seed{seed}", "seed": seed})
    result_path = RESULT_DIR / f"seed{seed}_control_results.npz"
    summary = pipe.run_atlas_deepc(H, labels, variant, save_path=result_path)
    data = np.load(result_path, allow_pickle=True)
    dist = np.asarray(data["distance"], dtype=float)
    summary["practical_success_1p05cm"] = bool(dist[-30:].mean() <= 0.0105 and dist[-30:].max() < 0.02)
    summary["result_path"] = str(result_path)
    summaries.append(summary)
    print(
        seed,
        "mean30", f"{summary['mean_last_30']:.5f}",
        "max30", f"{summary['max_last_30']:.5f}",
        "practical", summary["practical_success_1p05cm"],
        "chart sets", summary["unique_chart_sets"][:4],
    )

SUMMARY_PATH.write_text(json.dumps(summaries, indent=2))
summaries


## 5. Summary Plot


In [ ]:
summaries = json.loads(SUMMARY_PATH.read_text())
seeds = [s["seed"] for s in summaries]
mean30 = [s["mean_last_30"] for s in summaries]
max30 = [s["max_last_30"] for s in summaries]
mins = [s["min"] for s in summaries]

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(seeds))
ax.bar(x - 0.2, mean30, width=0.2, label="mean last 30")
ax.bar(x, max30, width=0.2, label="max last 30")
ax.bar(x + 0.2, mins, width=0.2, label="min")
ax.axhline(0.0105, color="tab:green", linestyle="--", linewidth=1, label="1.05 cm practical target")
ax.axhline(0.02, color="tab:red", linestyle="--", linewidth=1, label="2 cm max-tail bound")
ax.set_xticks(x, [str(s) for s in seeds])
ax.set_xlabel("Seed")
ax.set_ylabel("Distance")
ax.set_title("Overlap-hybrid atlas DeePC multi-target summary")
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
summary_fig = RESULT_DIR / "multi_seed_summary.png"
fig.savefig(summary_fig, dpi=180)
summary_fig


## 6. Tracking Diagnostics For One Seed


In [ ]:
PLOT_SEED = 1
result_path = RESULT_DIR / f"seed{PLOT_SEED}_control_results.npz"
data = np.load(result_path, allow_pickle=True)
y = data["y"]
pred = data["pred"]
ref = data["ref"]
u = data["u"]
dist = data["distance"]
steps = np.arange(len(dist))

fig, axes = plt.subplots(6, 1, figsize=(12, 14), sharex=True)
fig.suptitle(f"Overlap-hybrid Reacher tracking diagnostics, seed {PLOT_SEED}", fontsize=16)

items = [
    ("dq1", 4),
    ("dq2", 5),
    ("x_ee", 6),
    ("y_ee", 7),
]
for ax, (name, idx) in zip(axes[:4], items):
    ax.plot(steps, y[:, idx], label=f"{name} meas")
    ax.plot(steps, pred[:, idx], "--", label=f"{name} plan")
    ax.plot(steps, ref[:, idx], ":", label=f"{name} target")
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

axes[4].plot(steps, u[:, 0], label="u1")
axes[4].plot(steps, u[:, 1], label="u2")
axes[4].axhline(BASE_VARIANT["u_limit"], color="red", linestyle="--", alpha=0.6, label="upper limit")
axes[4].axhline(-BASE_VARIANT["u_limit"], color="red", linestyle="--", alpha=0.6, label="lower limit")
axes[4].set_ylabel("input")
axes[4].grid(True, alpha=0.3)
axes[4].legend(loc="best")

axes[5].plot(steps, dist, label="distance")
axes[5].axhline(0.0105, color="tab:green", linestyle="--", linewidth=1, label="1.05 cm")
axes[5].axhline(0.02, color="tab:red", linestyle="--", linewidth=1, label="2 cm")
axes[5].set_ylabel("distance")
axes[5].set_xlabel("Time step")
axes[5].grid(True, alpha=0.3)
axes[5].legend(loc="best")

fig.tight_layout()
tracking_fig = RESULT_DIR / f"seed{PLOT_SEED}_tracking.png"
fig.savefig(tracking_fig, dpi=180)
tracking_fig


## 7. Optional MuJoCo Video

Set `MAKE_VIDEO=True` to generate a MuJoCo video for one seed. This reruns control for that seed with `render_mode="rgb_array"`.


In [ ]:
MAKE_VIDEO = False
VIDEO_SEED = 1

if MAKE_VIDEO:
    variant = pipe.Variant(**{**BASE_VARIANT, "name": f"overlap_hybrid_seed{VIDEO_SEED}_video", "seed": VIDEO_SEED})
    pipe.run_atlas_deepc(
        H,
        labels,
        variant,
        save_path=RESULT_DIR / f"seed{VIDEO_SEED}_video_control_results.npz",
        video_path=RESULT_DIR / f"seed{VIDEO_SEED}_mujoco.mp4",
    )
    print("Saved", RESULT_DIR / f"seed{VIDEO_SEED}_mujoco.mp4")
else:
    print("Set MAKE_VIDEO=True to render a MuJoCo video.")
